In [0]:
# ════════════════════════════════════════════════════════════════
# run_framework_v2.py - OPTIMIZED FOR NEW SCHEMA
# ════════════════════════════════════════════════════════════════
# Supports:
#   - Unified schema: data_flow_l0_detail, data_flow_pb_detail
#   - All layers: L0 (ingestion), L1 (bronze), L2 (silver)
#   - Load types: FULL, INCREMENTAL, MERGE
#   - File formats: csv, json, parquet, delta, excel
# ════════════════════════════════════════════════════════════════

In [0]:
# PARAMETERS
dbutils.widgets.text("GROUP_ID", "")
dbutils.widgets.text("TARGET_LOAD_TABLE", "")
dbutils.widgets.text("ENVIRONMENT", "dev")

GROUP_ID = dbutils.widgets.get("GROUP_ID").strip().upper()
TARGET_TABLE = dbutils.widgets.get("TARGET_LOAD_TABLE").strip()
ENV = dbutils.widgets.get("ENVIRONMENT").strip()

# Determine layer from GROUP_ID suffix
if GROUP_ID.endswith("_L0"):
    LAYER = "L0"
elif GROUP_ID.endswith("_L1"):
    LAYER = "L1"
elif GROUP_ID.endswith("_L2"):
    LAYER = "L2"
else:
    LAYER = "ALL"

if not GROUP_ID:
    raise ValueError("GROUP_ID is required")

print(f"GROUP_ID      : {GROUP_ID}")
print(f"LAYER         : {LAYER}")
print(f"TARGET_TABLE  : {TARGET_TABLE or 'ALL'}")
print(f"ENVIRONMENT   : {ENV}")

In [0]:
# IMPORTS
import traceback
from datetime import datetime
from pyspark.sql import functions as F

# Hard-code catalog to avoid serverless permissions issues
CATALOG = "demo_catalog"

print(f"CATALOG       : {CATALOG}")

In [0]:
# READ SOURCE
def read_source(url, fmt="csv", delimiter=","):
    """
    Read from HTTP/S3/DBFS/Volumes into Spark DataFrame.
    Supports: csv, json, parquet, delta
    """
    fmt = (fmt or "csv").strip().lower()
    url = url.strip()
    
    if not url:
        raise ValueError("Source URL is empty")
    
    print(f"  Reading [{fmt}] from: {url[:80]}...")
    
    # HTTP/HTTPS sources
    if url.startswith("http"):
        import requests, io, pandas as pd
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        
        if fmt == "csv":
            pdf = pd.read_csv(io.BytesIO(response.content), sep=delimiter)
            return spark.createDataFrame(pdf)
        elif fmt == "json":
            pdf = pd.read_json(io.BytesIO(response.content))
            return spark.createDataFrame(pdf)
        elif fmt == "parquet":
            pdf = pd.read_parquet(io.BytesIO(response.content))
            return spark.createDataFrame(pdf)
        else:
            raise ValueError(f"Unsupported HTTP format: {fmt}")
    
    # Cloud storage paths
    else:
        if fmt == "csv":
            return spark.read.option("header", "true").option("inferSchema", "true").option("sep", delimiter).csv(url)
        elif fmt == "json":
            return spark.read.json(url)
        elif fmt == "parquet":
            return spark.read.parquet(url)
        elif fmt == "delta":
            return spark.read.format("delta").load(url)
        else:
            raise ValueError(f"Unsupported format: {fmt}")

In [0]:
# WRITE TABLE
def write_table(df, catalog, schema, table, load_type="FULL", merge_keys=None):
    """
    Write DataFrame to Delta table.
    Returns row count.
    """
    # Create schema if needed
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    
    full_name = f"{catalog}.{schema}.{table}"
    load_type = (load_type or "FULL").strip().upper()
    
    if load_type == "FULL":
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
        return df.count()
    
    elif load_type in ("INCREMENTAL", "APPEND"):
        df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_name)
        return df.count()
    
    elif load_type == "MERGE":
        if not merge_keys:
            raise ValueError(f"MERGE_KEYS required for MERGE load type")
        
        keys = [k.strip() for k in merge_keys.split(",")]
        temp_view = f"_tmp_{table}"
        df.createOrReplaceTempView(temp_view)
        
        # Create table if not exists
        spark.sql(f"CREATE TABLE IF NOT EXISTS {full_name} USING DELTA AS SELECT * FROM {temp_view} WHERE 1=0")
        
        # Build MERGE statement
        on_clause = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        update_cols = [c for c in df.columns if c not in keys]
        update_set = ", ".join([f"target.{c} = source.{c}" for c in update_cols])
        insert_cols = ", ".join(df.columns)
        insert_vals = ", ".join([f"source.{c}" for c in df.columns])
        
        merge_sql = f"""
            MERGE INTO {full_name} AS target
            USING {temp_view} AS source
            ON {on_clause}
            WHEN MATCHED THEN UPDATE SET {update_set}
            WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
        """
        spark.sql(merge_sql)
        return df.count()
    
    else:
        raise ValueError(f"Unsupported load type: {load_type}")

In [0]:
# AUDIT LOG
def write_audit(group_id, table_name, layer, status, message, rows, start_time, end_time):
    """
    Write audit record. Never raises.
    """
    try:
        safe_msg = str(message).replace("'", "''")[:400]
        start_ts = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_ts = end_time.strftime("%Y-%m-%d %H:%M:%S")
        
        spark.sql(f"""
            INSERT INTO {CATALOG}.admin.audit_log (
                DATA_FLOW_GROUP_ID,
                TARGET_TABLE,
                STATUS,
                MESSAGE,
                ETL_LAYER,
                ROWS_PROCESSED,
                START_TIME,
                END_TIME,
                LOAD_TS
            )
            VALUES (
                '{group_id}',
                '{table_name}',
                '{status}',
                '{safe_msg}',
                '{layer}',
                {rows},
                '{start_ts}',
                '{end_ts}',
                current_timestamp()
            )
        """)
    except Exception as e:
        print(f"  ⚠ Audit write failed: {e}")

In [0]:
# PROCESS LAYER
def process_layer(detail_table, layer):
    """
    Process all objects for a given layer.
    """
    print(f"\n{'='*60}")
    print(f"  LAYER: {layer} | TABLE: {detail_table}")
    print(f"{'='*60}")
    
    # Build query based on layer
    if layer == "L0":
        obj_col = "SOURCE_OBJ_NAME"
        table_filter = f"AND SOURCE_OBJ_NAME = '{TARGET_TABLE}'" if TARGET_TABLE and TARGET_TABLE.upper() != "ALL" else ""
        layer_filter = ""
    else:
        obj_col = "TARGET_OBJ_NAME"
        table_filter = f"AND TARGET_OBJ_NAME = '{TARGET_TABLE}'" if TARGET_TABLE and TARGET_TABLE.upper() != "ALL" else ""
        layer_filter = f"AND ETL_LAYER = '{layer}'"
    
    # Query control table
    query = f"""
        SELECT * 
        FROM {CATALOG}.admin.{detail_table}
        WHERE DATA_FLOW_GROUP_ID = '{GROUP_ID}'
          AND IS_ACTIVE = 'Y'
          {layer_filter}
          {table_filter}
        ORDER BY {obj_col}
    """
    
    rows = spark.sql(query).collect()
    
    if not rows:
        print(f"  ⚠ No active objects found for {layer}")
        return True
    
    print(f"  Objects to process: {len(rows)}\n")
    
    success = True
    
    for row in rows:
        r = row.asDict()
        t0 = datetime.now()
        status = "FAILED"
        msg = ""
        count = 0
        
        try:
            if layer == "L0":
                # ═══════════════════════════════════════════════════
                # L0: File Ingestion
                # ═══════════════════════════════════════════════════
                source_url = (r.get("SOURCE") or "").strip()
                target_schema = (r.get("SOURCE_OBJ_SCHEMA") or "").strip()
                target_table = (r.get("SOURCE_OBJ_NAME") or "").strip()
                file_format = (r.get("INPUT_FILE_FORMAT") or "csv").strip()
                load_type = (r.get("LOAD_TYPE") or "FULL").strip()
                delimiter = (r.get("DELIMETER") or ",").strip()
                
                # Strip file extension from table name
                import os
                target_table = os.path.splitext(target_table)[0]
                
                full_name = f"{CATALOG}.{target_schema}.{target_table}"
                
                print(f"  ▶ {full_name}")
                print(f"    Source: {source_url[:60]}...")
                print(f"    Format: {file_format} | Load: {load_type}")
                
                # Read source
                df = read_source(source_url, file_format, delimiter)
                
                # Add audit columns
                df = (
                    df
                    .withColumn("_etl_group_id", F.lit(GROUP_ID))
                    .withColumn("_etl_layer", F.lit(layer))
                    .withColumn("_etl_env", F.lit(ENV))
                    .withColumn("_etl_load_ts", F.current_timestamp())
                )
                
                # Write
                count = write_table(df, CATALOG, target_schema, target_table, load_type)
                status = "SUCCESS"
                msg = f"{count:,} rows"
                print(f"    ✅ {msg}")
                
            else:
                # ═══════════════════════════════════════════════════
                # L1/L2: Transformation
                # ═══════════════════════════════════════════════════
                target_schema = (r.get("TARGET_OBJ_SCHEMA") or "").strip()
                target_table = (r.get("TARGET_OBJ_NAME") or "").strip()
                transform_query = (r.get("TRANSFORM_QUERY") or r.get("TRANSFORMATION_QUERY") or "").strip()
                load_type = (r.get("LOAD_TYPE") or "FULL").strip()
                merge_keys = (r.get("MERGE_KEYS") or "").strip()
                source_schema = (r.get("SOURCE_OBJ_SCHEMA") or "").strip()
                source_table = (r.get("SOURCE_OBJ_NAME") or "").strip()
                
                full_name = f"{CATALOG}.{target_schema}.{target_table}"
                
                print(f"  ▶ {full_name}")
                print(f"    Load: {load_type}")
                
                # Execute transformation or read source table
                if transform_query:
                    # Add catalog prefix if missing
                    if source_schema and f"{source_schema}." in transform_query and f"{CATALOG}.{source_schema}." not in transform_query:
                        transform_query = transform_query.replace(f"{source_schema}.", f"{CATALOG}.{source_schema}.")
                    
                    df = spark.sql(transform_query)
                else:
                    df = spark.table(f"{CATALOG}.{source_schema}.{source_table}")
                
                # Add audit columns
                df = (
                    df
                    .withColumn("_etl_group_id", F.lit(GROUP_ID))
                    .withColumn("_etl_layer", F.lit(layer))
                    .withColumn("_etl_env", F.lit(ENV))
                    .withColumn("_etl_load_ts", F.current_timestamp())
                )
                
                # Write
                count = write_table(df, CATALOG, target_schema, target_table, load_type, merge_keys)
                status = "SUCCESS"
                msg = f"{count:,} rows"
                print(f"    ✅ {msg}")
        
        except Exception as e:
            success = False
            status = "FAILED"
            msg = f"{type(e).__name__}: {str(e)[:300]}"
            print(f"    ❌ FAILED: {msg}")
        
        finally:
            t1 = datetime.now()
            duration = (t1 - t0).total_seconds()
            print(f"    ⏱ Duration: {duration:.1f}s\n")
            
            table_name = target_table if layer == "L0" else target_table
            write_audit(GROUP_ID, table_name, layer, status, msg, count, t0, t1)
    
    return success

In [0]:
# MAIN EXECUTION
start_time = datetime.now()

print(f"\n{'╔'+'═'*58+'╗'}")
print(f"║  ETL FRAMEWORK v2.0 - START                          ║")
print(f"║  {GROUP_ID:<50}  ║")
print(f"{'╚'+'═'*58+'╝'}\n")

try:
    all_success = True
    
    if LAYER == "L0":
        all_success = process_layer("data_flow_l0_detail", "L0")
    
    elif LAYER == "L1":
        all_success = process_layer("data_flow_pb_detail", "L1")
    
    elif LAYER == "L2":
        all_success = process_layer("data_flow_pb_detail", "L2")
    
    elif LAYER == "ALL":
        all_success = process_layer("data_flow_l0_detail", "L0") and all_success
        all_success = process_layer("data_flow_pb_detail", "L1") and all_success
        all_success = process_layer("data_flow_pb_detail", "L2") and all_success
    
    else:
        raise ValueError(f"Invalid LAYER: {LAYER}")
    
    end_time = datetime.now()
    total_duration = (end_time - start_time).total_seconds()
    
    print(f"\n{'╔'+'═'*58+'╗'}")
    if all_success:
        print(f"║  ✅ SUCCESS - All objects processed                   ║")
    else:
        print(f"║  ❌ FAILED - Some objects failed                      ║")
    print(f"║  Duration: {total_duration:.1f}s{' '*(43-len(str(total_duration)))}║")
    print(f"{'╚'+'═'*58+'╝'}\n")
    
    if not all_success:
        raise Exception(
            f"Pipeline failed for GROUP_ID={GROUP_ID}. "
            f"Check: SELECT * FROM {CATALOG}.admin.audit_log "
            f"WHERE DATA_FLOW_GROUP_ID='{GROUP_ID}' ORDER BY LOAD_TS DESC"
        )

except Exception as e:
    print(f"\n❌ PIPELINE FAILED: {e}")
    raise